<a href="https://colab.research.google.com/github/lseidy/BI-Data-Sience/blob/main/preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 2 — Exercício 1: Preprocessing nas reviews

Hoje vocês vão aplicar 3 técnicas clássicas de preprocessing (tokenização, stop words, stemming) nas mesmas 25 reviews da aula 1, e ver o que acontece.

## Como funciona esse notebook

- O código já vem pronto, com comentários explicando cada linha.
- Em **2 momentos** vocês fazem uma modificação pequena (trocar uma palavra, mudar um parâmetro). Esses momentos estão marcados com 🟧 e têm instruções literais ("apague isso, digite aquilo").
- Não precisa escrever função inteira. Não precisa decorar regex.
- Roda as células em ordem (botão ▶ ou Shift+Enter).

## Tempo: 30 minutos

Quem terminar antes, dá uma olhada na seção **"Aprofundamento (opcional)"** no fim — tem ideias pra continuar em casa.

## Setup

Roda essa célula uma vez. Instala o NLTK (biblioteca clássica de PLN em Python) e baixa os recursos que vamos usar.

In [ ]:
import nltk
nltk.download("stopwords", quiet=True)
nltk.download("rslp", quiet=True)

import re
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer

print("Setup pronto.")

Setup pronto.


## As 25 reviews (aula 1)

Mesmo dataset. Não precisa mexer nessa célula — só roda.

In [ ]:
reviews = [
    "Chegou em 20min, ainda quentinho. Entregador super educado. Nota 10!",
    "Péssimo. Esperei 1h40 e quando chegou veio errado. Nunca mais uso.",
    "App fácil de usar, mas o preço do frete tá abusivo. Restaurante bom porém.",
    "Ah sim, claro, adorei esperar 2h por um X-burguer frio. Serviço impecável.",
    "Deu ruim",
    "Mano, tava com saudade desse Uber Eats aqui não kkkk RápidoJá é outro nível, salvou meu rolê",
    "Entrega rápida, atendimento ok, comida chegou morna. No geral, tá valendo.",
    "Três tentativas de pagamento e nada. Suporte não responde. Lixo de app.",
    "Amo os cupons de desconto! Sempre tem promoção boa aqui.",
    "Pedi duas Coca, veio uma. Reclamei, devolveram o valor em crédito na hora. Resolveu.",
    "Interface confusa, demorei pra achar onde aplicar o cupom. Mas a comida tava ótima.",
    "Entregador sumiu com meu pedido. Só mostrou 'entregue' no app. Estou processando.",
    "tudo certo",
    "Ótima empresa, se você gosta de receber pedido errado e sem reembolso",
    "Uso todo dia, nunca tive problema. Recomendo pros amigos.",
    "Entrega veio antes do previsto e a embalagem tava impecável. Parabéns!",
    "Cobraram duas vezes no cartão e levou uma semana pra estornar. Inadmissível.",
    "Prático, resolve. Não tenho do que reclamar.",
    "Só fazendo propaganda enganosa mesmo. Tempo estimado de 30min virou 1h30.",
    "Não é ruim.",
    "Bah, tri de boa pra pedir no almoço, guri. Chegou quentinho e o preço é de lei.",
    "Comida excelente, entrega horrível. Difícil escolher nota.",
    "Zero problema, experiência tranquila.",
    "Cancela minha conta. Não quero mais usar isso aqui nunca mais.",
    "Maravilhoso, se o objetivo era me fazer perder a fome.",
]

print(f"{len(reviews)} reviews carregadas.")

25 reviews carregadas.


---

## Parte 1 — Tokenização

Tokenização = quebrar texto em pedaços (tokens). A função abaixo faz isso em 2 passos.

Roda a célula e olha o output.

In [ ]:
def tokenizar(texto):
    # Passo 1: deixa tudo em minúsculas ("Chegou" e "chegou" viram a mesma coisa)
    texto_minusculo = texto.lower()

    # Passo 2: encontra todas as sequências de letras/números do texto.
    # \w+ é uma expressão que casa com "uma ou mais letras/números".
    # Em Python 3, isso já entende acentos (á, ç, etc.).
    tokens = re.findall(r"\w+", texto_minusculo)

    return tokens


# Demo: ver o resultado em 3 frases
exemplos = [
    "Olá, mundo!",
    "Chegou em 20min, ainda quentinho.",
    "Maravilhoso, se o objetivo era me fazer perder a fome.",
]
for frase in exemplos:
    print(f"Original: {frase}")
    print(f"Tokens:   {tokenizar(frase)}\n")

Original: Olá, mundo!
Tokens:   ['olá', 'mundo']

Original: Chegou em 20min, ainda quentinho.
Tokens:   ['chegou', 'em', '20min', 'ainda', 'quentinho']

Original: Maravilhoso, se o objetivo era me fazer perder a fome.
Tokens:   ['maravilhoso', 'se', 'o', 'objetivo', 'era', 'me', 'fazer', 'perder', 'a', 'fome']



Aplicando nas 25 reviews:

In [ ]:
tokens_por_review = [tokenizar(r) for r in reviews]
total_tokens_original = sum(len(t) for t in tokens_por_review)

print(f"Total de tokens nas 25 reviews: {total_tokens_original}")
print(f"\nReview #1: {tokens_por_review[0]}")

Total de tokens nas 25 reviews: 256

Review #1: ['chegou', 'em', '20min', 'ainda', 'quentinho', 'entregador', 'super', 'educado', 'nota', '10']


### 🟧 Modificação 1 — Filtrar tokens muito curtos

Reparou que a tokenização inclui palavras de 1 ou 2 letras (como `a`, `o`, `é`)? Em muitas análises essas palavras quase não trazem informação. Vamos modificar a função pra **ignorar qualquer token com menos de 3 letras**.

#### O que você vai mudar

Na célula abaixo, dentro de uma linha tem um trecho `\w+` entre aspas. Esse trecho é uma "regra de busca" que diz **"1 ou mais letras"**. Vamos trocar por `\w{3,}`, que diz **"3 ou mais letras"**.

Não precisa entender regex pra fazer essa mudança — só precisa saber a diferença:

| Trecho | Significado |
|---|---|
| `\w+`   | 1 ou mais letras/números |
| `\w{3,}` | 3 ou mais letras/números |

#### Como mudar (passo a passo)

1. Na célula abaixo, encontre a linha que tem `re.findall(r"\w+", texto_minusculo)`.
2. Apague o `+` e digite `{3,}` no lugar.
3. A linha final fica: `re.findall(r"\w{3,}", texto_minusculo)`.
4. Roda a célula (▶) e veja a comparação no output.

Se travar, descomenta a célula "🔓 Solução" mais abaixo.

In [ ]:
def tokenizar_v2(texto):
    texto_minusculo = texto.lower()

    # 🟧 SUA MUDANÇA AQUI EMBAIXO 🟧
    #
    # Antes:  tokens = re.findall(r"\w+",   texto_minusculo)
    # Depois: tokens = re.findall(r"\w{3,}", texto_minusculo)
    #
    # Apague o "+" e digite "{3,}" no lugar (sem mudar mais nada na linha):
    tokens = re.findall(r"\w{3,}", texto_minusculo)

    return tokens


# Comparar antes/depois (não precisa mexer aqui)
tokens_v2 = [tokenizar_v2(r) for r in reviews]
total_v2 = sum(len(t) for t in tokens_v2)

print(f"Versão original (1+ letras): {total_tokens_original} tokens")
print(f"Sua versão (3+ letras):      {total_v2} tokens")
print(f"Diferença: {total_tokens_original - total_v2} tokens removidos")
print(f"\nReview #1 antes: {tokens_por_review[0]}")
print(f"Review #1 depois: {tokens_v2[0]}")

if total_tokens_original == total_v2:
    print("\n👀 Diferença = 0. Você ainda não fez a mudança — leia as instruções acima.")

Versão original (1+ letras): 256 tokens
Sua versão (3+ letras):      207 tokens
Diferença: 49 tokens removidos

Review #1 antes: ['chegou', 'em', '20min', 'ainda', 'quentinho', 'entregador', 'super', 'educado', 'nota', '10']
Review #1 depois: ['chegou', '20min', 'ainda', 'quentinho', 'entregador', 'super', 'educado', 'nota']


#### 🔓 Solução Modificação 1

*Tenta antes de olhar.*

Se travou, descomenta a célula abaixo (apaga os `#` do começo das linhas) e roda.

In [ ]:
# def tokenizar_v2(texto):
#     texto_minusculo = texto.lower()
#     tokens = re.findall(r"\w{3,}", texto_minusculo)
#     return tokens

**Pergunta de reflexão:** o que sumiu do output? Foi bom ou ruim? Em qual cenário você usaria essa versão e em qual não usaria?

---

## Parte 2 — Stop words

Stop words = palavras frequentes mas pouco informativas. O NLTK já tem uma lista pronta pra português.

In [ ]:
stop_words_padrao = set(stopwords.words("portuguese"))

print(f"Lista padrão tem {len(stop_words_padrao)} palavras.")
print("Algumas:", sorted(stop_words_padrao)[:30])
print("\n⚠️  Repara: 'não' está na lista. Lembra do problema na aula 1?")

Lista padrão tem 207 palavras.
Algumas: ['a', 'ao', 'aos', 'aquela', 'aquelas', 'aquele', 'aqueles', 'aquilo', 'as', 'até', 'com', 'como', 'da', 'das', 'de', 'dela', 'delas', 'dele', 'deles', 'depois', 'do', 'dos', 'e', 'ela', 'elas', 'ele', 'eles', 'em', 'entre', 'era']

⚠️  Repara: 'não' está na lista. Lembra do problema na aula 1?


### 🟧 Modificação 2 — Adicionar palavras à lista personalizada

A lista padrão do NLTK é genérica. Nas nossas reviews aparecem palavras de gíria/internet (`kkkk`, `mesmo`, `ainda`) que poluem o resultado mas não estão na lista padrão. Vamos adicionar mais 2 palavras à nossa lista personalizada.

#### O que você vai mudar

Na célula abaixo há uma lista chamada `minhas_stop_words_extras` com **3 palavras já preenchidas**. Você precisa adicionar mais **2**.

#### Como mudar (passo a passo)

1. Escolha 2 palavras das sugestões: `"kkkk"`, `"mesmo"`, `"ainda"`, `"super"`, `"pros"`.
2. Adicione cada uma numa linha nova, **dentro de aspas duplas**, **com vírgula no final**.
3. Mantenha o **recuo de 4 espaços** no início da linha (Python é sensível a isso).
4. Apague o comentário `# 🟧 ADICIONE...` quando terminar (ou deixa, tanto faz).

#### Exemplo visual

Se você quisesse adicionar `"kkkk"`, ficaria assim:

```python
minhas_stop_words_extras = [
    "pra",
    "tá",
    "aí",
    "kkkk",      # ← linha nova: aspas + palavra + aspas + vírgula
    # 🟧 ADICIONE MAIS 1 PALAVRA AQUI
]
```

In [ ]:
minhas_stop_words_extras = [
    "pra",
    "tá",
    "aí",
    "Bah"
    # 🟧 ADICIONE 2 PALAVRAS AQUI (uma por linha, entre aspas duplas, com vírgula no final).
    # Sugestões: "kkkk", "mesmo", "ainda", "super", "pros".
    # Mantenha 4 espaços de recuo antes das aspas (igual às linhas acima).
]

stop_words_completo = stop_words_padrao | set(minhas_stop_words_extras)
print(f"Lista final: {len(stop_words_completo)} palavras (+{len(minhas_stop_words_extras)} suas)")

if len(minhas_stop_words_extras) < 5:
    print(f"\n👀 Você adicionou {len(minhas_stop_words_extras)} palavras na lista. A meta é 5 (3 já estavam, faltam {5 - len(minhas_stop_words_extras)}).")

Lista final: 211 palavras (+4 suas)

👀 Você adicionou 4 palavras na lista. A meta é 5 (3 já estavam, faltam 1).


In [ ]:
def remover_stop_words(tokens):
    return [t for t in tokens if t not in stop_words_completo]

tokens_sem_stop = [remover_stop_words(t) for t in tokens_por_review]
total_sem_stop = sum(len(t) for t in tokens_sem_stop)

print(f"Antes:  {total_tokens_original} tokens")
print(f"Depois: {total_sem_stop} tokens")
print(f"Redução: {100*(total_tokens_original - total_sem_stop)/total_tokens_original:.0f}%")

Antes:  256 tokens
Depois: 180 tokens
Redução: 30%


### A armadilha do `não`

Olha o que aconteceu com a review #20 (`"Não é ruim."`):

In [ ]:
i = 19  # review #20 (índice começa em 0)
print(f"Original:        {reviews[i]!r}")
print(f"Tokens:          {tokens_por_review[i]}")
print(f"Sem stop words:  {tokens_sem_stop[i]}")
print("\n👉 Sentido positivo (\"não é ruim\" = elogio brando) virou só [\"ruim\"].")
print("   Inverteu o sinal. Preprocessing não é gratuito.")

Original:        'Não é ruim.'
Tokens:          ['não', 'é', 'ruim']
Sem stop words:  ['ruim']

👉 Sentido positivo ("não é ruim" = elogio brando) virou só ["ruim"].
   Inverteu o sinal. Preprocessing não é gratuito.


---

## Parte 3 — Stemming

**Stemming** = corte mecânico das terminações. *correndo → corr*. Rápido, errado, mas suficiente em busca.

**Lematização** = forma canônica via dicionário. *correndo → correr*. Lento, certo, exige biblioteca pesada (spaCy).

Pra esse exercício vamos usar o **stemmer RSLP** do NLTK (específico pra português). Não tem modificação aqui — só rode.

In [ ]:
stemmer = RSLPStemmer()

# Demo: ver como diferentes formas de uma palavra ficam parecidas após stemming
exemplos = ["correndo", "correu", "corremos", "correr", "casas", "casa", "adorei", "adoramos"]
for palavra in exemplos:
    print(f"{palavra:15} → {stemmer.stem(palavra)}")

correndo        → corr
correu          → corr
corremos        → corr
correr          → corr
casas           → cas
casa            → cas
adorei          → ador
adoramos        → ador


In [ ]:
def aplicar_stemming(tokens):
    return [stemmer.stem(t) for t in tokens]

tokens_stemmed = [aplicar_stemming(t) for t in tokens_sem_stop]

print("Review #1")
print(f"Original:    {tokens_por_review[0]}")
print(f"Sem stop:    {tokens_sem_stop[0]}")
print(f"+ stemming:  {tokens_stemmed[0]}")

Review #1
Original:    ['chegou', 'em', '20min', 'ainda', 'quentinho', 'entregador', 'super', 'educado', 'nota', '10']
Sem stop:    ['chegou', '20min', 'ainda', 'quentinho', 'entregador', 'super', 'educado', 'nota', '10']
+ stemming:  ['cheg', '20min', 'aind', 'quent', 'entreg', 'sup', 'educ', 'not', '10']


---

## Parte 4 — Comparar tudo

Tabela final mostrando quantos tokens cada review tinha em cada etapa. Só rode.

In [ ]:
import pandas as pd

tabela = pd.DataFrame({
    "#": range(1, 26),
    "original": [len(t) for t in tokens_por_review],
    "sem_stop_words": [len(t) for t in tokens_sem_stop],
    "+ stemming (únicos)": [len(set(t)) for t in tokens_stemmed],
})
tabela["redução_%"] = (
    100 * (tabela["original"] - tabela["+ stemming (únicos)"]) / tabela["original"]
).round(0)
tabela

,#,original,sem_stop_words,+ stemming (únicos),redução_%
0,1,10,9,9,10.0
1,2,11,8,8,27.0
2,3,14,9,9,36.0
3,4,13,11,11,15.0
4,5,2,2,2,0.0
5,6,17,13,13,24.0
6,7,11,9,9,18.0
7,8,12,8,8,33.0
8,9,10,7,7,30.0
9,10,14,10,10,29.0


In [ ]:
print(f"Tokens originais:        {tabela['original'].sum()}")
print(f"Após stop words:         {tabela['sem_stop_words'].sum()}")
print(f"Após stemming (únicos):  {tabela['+ stemming (únicos)'].sum()}")
print(f"\nRedução média por review: {tabela['redução_%'].mean():.0f}%")

Tokens originais:        256
Após stop words:         180
Após stemming (únicos):  179

Redução média por review: 28%


---

## Reflexão

Respondam:

1. **Qual review teve o sentido invertido** depois do preprocessing? *(Dica: olha a #20.)*
2. **Em qual cenário** vocês confiariam nesse pipeline? Em qual *não* confiariam?
3. Se o objetivo final é **alimentar um LLM**, vale a pena rodar tudo isso? *(Spoiler: nem sempre. Vamos discutir nos slides 17-18.)*

---

## (Bônus) Contagem de tokens estilo LLM

Os LLMs **não** usam tokenização baseada em palavra como a nossa. Usam BPE (Byte-Pair Encoding), que quebra em sub-palavras. Roda essa célula pra ver a diferença.

In [ ]:
%pip install -q tiktoken

import tiktoken
enc = tiktoken.get_encoding("cl100k_base")

exemplo = "Chegou em 20min, ainda quentinho. Entregador super educado. Nota 10!"

print(f"Texto: {exemplo!r}\n")
print(f"Tokenização nossa (palavras):  {len(tokenizar(exemplo))} tokens")
print(f"  → {tokenizar(exemplo)}\n")

tokens_llm = enc.encode(exemplo)
print(f"Tokenização LLM (BPE):         {len(tokens_llm)} tokens")
print(f"  → {[enc.decode([t]) for t in tokens_llm]}")

Texto: 'Chegou em 20min, ainda quentinho. Entregador super educado. Nota 10!'

Tokenização nossa (palavras):  10 tokens
  → ['chegou', 'em', '20min', 'ainda', 'quentinho', 'entregador', 'super', 'educado', 'nota', '10']

Tokenização LLM (BPE):         25 tokens
  → ['Che', 'g', 'ou', ' em', ' ', '20', 'min', ',', ' ainda', ' qu', 'ent', 'inho', '.', ' Ent', 'reg', 'ador', ' super', ' educ', 'ado', '.', ' Not', 'a', ' ', '10', '!']


**Observa:** o LLM "vê" mais tokens que palavras — porque quebra em sub-palavras. É por isso que ele cobra por token, não por palavra. **Texto preprocessado = menos tokens = conta menor.**

---

# 🌱 Aprofundamento (opcional, pra casa)

Se você curtiu, aqui ficam 3 ideias pra continuar explorando depois da aula. Tudo opcional. Não tem entrega.

## Antes de começar: como descomentar código

Os starters abaixo vêm **comentados** (com `#` no começo de cada linha). O Python ignora linhas que começam com `#`.

Pra ativar o código, você precisa **apagar o `#` e o espaço que vem depois** no começo de cada linha.

**Atalho do Colab:** seleciona as linhas que quer descomentar e aperta `Ctrl + /` (Windows/Linux) ou `Cmd + /` (Mac). Descomenta tudo de uma vez.

### Ideia 1 — Tokenização que entende emojis e hashtags

Nossa tokenização ignora `kkkk`, `👍`, `#delivery`. Em apps reais (rede social, chat de suporte), esses sinais carregam muita informação.

**O que o starter faz:** usa um regex mais sofisticado que captura **palavras OU hashtags OU emojis**.

**Pra rodar:** descomenta as 3 linhas abaixo e veja o output.

In [ ]:
# Starter — descomentar pra rodar:

# regex_inteligente = r"#\w+|[\U0001F300-\U0001FAFF]|\w+"
# texto_teste = "Chegou em 20min #rapidoja 👍 muito bom kkkk"
# print(re.findall(regex_inteligente, texto_teste.lower()))

### Ideia 2 — Preservar negação

O grande problema do nosso pipeline é que `"não é bom"` perde o `"não"` e fica só `["bom"]`. Uma técnica clássica: **juntar a negação à próxima palavra** antes de remover stop words. Ex: `"não é bom"` → `["NEG_é", "bom"]`.

**O que o starter faz:** define uma função `marcar_negacao` que detecta `"não"` antes de uma palavra e marca essa palavra como negada (prefixo `NEG_`).

**Pra rodar:** descomenta o bloco inteiro e veja o output nos 3 testes.

In [ ]:
# Starter — descomentar pra rodar:

# def marcar_negacao(tokens):
#     resultado = []
#     pular_proximo = False
#     for i, token in enumerate(tokens):
#         if pular_proximo:
#             pular_proximo = False
#             continue
#         if token == "não" and i + 1 < len(tokens):
#             resultado.append(f"NEG_{tokens[i+1]}")
#             pular_proximo = True
#         else:
#             resultado.append(token)
#     return resultado
#
# print(marcar_negacao(["não", "é", "bom"]))
# print(marcar_negacao(["comida", "boa"]))
# print(marcar_negacao(tokenizar(reviews[19])))  # "Não é ruim"

### Ideia 3 — Comparar com um tokenizador profissional

Nossa tokenização é simples. Bibliotecas como **spaCy** fazem muito mais: separam contrações (`"do"` → `["de", "o"]`), reconhecem entidades (datas, nomes próprios), fazem **lematização de verdade** (não só stemming).

**O que o starter faz:** instala o spaCy + modelo de português, processa uma review e mostra cada token com seu lema e categoria gramatical (POS).

**Pra rodar:** descomenta o bloco. ⚠️ A instalação demora ~1 min.

In [ ]:
# Starter — descomentar pra rodar (instalação demora ~1min):

# !pip install -q spacy
# !python -m spacy download pt_core_news_sm -q
#
# import spacy
# nlp = spacy.load("pt_core_news_sm")
#
# texto = reviews[0]
# doc = nlp(texto)
#
# print(f"Texto: {texto}\n")
# print("Token         | Lema          | POS")
# print("-" * 45)
# for token in doc:
#     print(f"{token.text:13} | {token.lemma_:13} | {token.pos_}")

---

Curtiu alguma ideia? Manda mensagem com o que descobriu, ou traz uma pergunta na próxima aula.

# Email Reclamação (para atividade 2)

In [ ]:
email = """
**Assunto:** Pedido #8419 não recebido

Boa noite,

Estou entrando em contato porque o pedido que fiz ontem à noite (terça, 27/04, por volta das 19h30) — número #8419, no Sushi Pelotas — apareceu como "entregue" no app às 22h05, mas nunca chegou aqui em casa. Conferi com o porteiro e com os vizinhos do andar, ninguém recebeu nada.

Tentei resolver pelo chat assim que percebi. O atendimento automático não entendeu o problema, fiquei uns 15 minutos girando em opções que não resolviam, até finalmente cair pra atendimento humano. O Bruno (acho que era esse o nome) foi educado, disse que ia verificar com o entregador e me deixou esperando uns 20 minutos sem retorno. Quando voltou, só falou que tinha sido "um equívoco no sistema" e que iam abrir uma investigação interna.

O problema é que o valor de R$ 184,90 já foi cobrado no meu cartão de crédito e até agora ninguém me passou prazo de estorno. E ainda tive que pedir comida em outro aplicativo na hora, porque tinha gente em casa esperando jantar.

Gostaria de pedir:
- Estorno integral dos R$ 184,90
- Algum cupom de cortesia pelo transtorno
- Um retorno deste email com prazo claro de devolução

Sou cliente do RápidoJá há um bom tempo e quase nunca tive problema, mas esse caso ficou mal resolvido e queria entender o que aconteceu.

Aguardo retorno.

Obrigada,
Camila Mendes
(53) 9XXXX-XXXX
"""Pedido não recebido
apareceu "entregue" nunca chegou
valor cobrado

# Reviews RápidoJá (para APS 1)

In [ ]:
reviews = [
    "Pedido atrasou 1h20. Comida chegou fria, ninguém merece.",
    "Cadê meu pedido? Marcou entregue há meia hora e nada.",
    "Suporte resolveu o ressarcimento em 5 minutos. Surpreso, sinceramente.",
    "Pizza do Boteco do Zé chegou quentinha e a massa tava perfeita. Pediria de novo amanhã.",
    "App travou 3 vezes na hora de finalizar o pedido. Tive que reiniciar tudo.",
    "Taxa de entrega de R$ 14 pra 2km de distância. Tá de brincadeira.",
    "Sushi chegou em 35 minutos, fresco e bem embalado. Nota dez.",
    "Atrasou de novo. Toda vez é a mesma história e o app não dá nem retorno.",
    "Atendente do chat super educada, resolveu meu problema com paciência.",
    "Hambúrguer artesanal do Burguer House é viciante. Sempre peço o mesmo combo.",
    "Pagamento não passou no app, mas o cartão foi debitado. Tive que abrir chamado e ninguém me retornou ainda.",
    "Pedido cancelado pelo restaurante depois de 1h esperando. Nem aviso teve.",
    "Cupom de aniversário valeu a pena. Pizza grande por menos da metade do preço.",
    "Comida fria de novo. O pastel veio molenga, parecia que tava na chuva.",
    "Suporte humano, não chatbot. Diferencial em 2026, sinceramente.",
    "Pedi às 19h, chegou às 21h30. Inaceitável pra um pedido a 4 quarteirões.",
    "Marmita do almoço estava ótima, arroz no ponto e a carne suculenta.",
    "App não consigo logar há 2 dias. Já reinstalei, mesma coisa.",
    'Promoção de "frete grátis" só funciona acima de R$ 80. Difícil pra um casal jantar.',
    "Pedido sumiu no meio da entrega. App ficou confuso, suporte só mandou aguardar.",
    "Açaí derreteu. 50 minutos pra chegar uma coisa que era pra estar gelada.",
    "Atendimento via chat foi rápido e simpático. Resolveu na hora.",
    "Comida do Sushi Niwa é o melhor da região. Sempre fresco, sempre certo.",
    "Cobrança duplicada no cartão. Tive que cancelar pelo banco e o app não me devolveu nada ainda.",
    "Hambúrguer ótimo, mas o entregador derramou metade da batata frita no caminho.",
    "Tudo certo hoje. Entrega no horário, comida boa, atendimento sem queixas.",
    "Demoraram tanto que tive que jantar outra coisa. Cobraram do mesmo jeito.",
    "Pizza fria. O motoboy disse que rodou bastante até achar o endereço.",
    "Pedido errado. Mandaram do outro cliente. Atendimento resolveu rápido, mas que perda de tempo.",
    "Aplicativo super fácil de usar. Pedido em 3 cliques.",
    "Cupons da home raramente funcionam de verdade. Maioria expirou ou tem letra miúda.",
    "Marmita chegou amassada. O motoboy provavelmente colocou outra coisa em cima na bag.",
    "Comida do Verdurama é leve e bem temperada. Bom pro almoço da semana toda.",
    "Atrasou mas o entregador foi gentil e pediu desculpa. Tolerável dessa vez.",
    "Custo-benefício péssimo sem cupom. Com cupom, dá pra justificar pedir.",
    'Filtro "sem glúten" não funciona, mostra restaurantes que não têm a opção.',
    "Comida do Tia Nilda é confortante. Lembra a casa da minha avó.",
    "Entrega rápida hoje. Em 25 minutos chegou. Surpreendi.",
    "Suporte transferiu minha reclamação 3 vezes pra atendentes diferentes. Frustrante demais.",
    "Sushi do Niwa fresco como sempre. Recomendo o combinado executivo.",
    "Pedido marcado como entregue, mas nunca chegou. Tive que abrir reclamação e perdi a noite.",
    "Pizza grande chegou fria, mas a borda recheada salvou. Voltaria a pedir.",
    "App responsivo, abre rápido, não trava. Sem queixas técnicas hoje.",
    "Notificações chegam quando o pedido já tá frio. Inúteis na prática.",
    "Atendente do chat foi educadíssima. Mereceu nota 10.",
    "1h45 pra um pedido a 3km. Nunca mais peço aqui.",
    "Comida sem reclamação, entrega no horário. Funcionou como deveria.",
    "Cardápio desatualizado no app. Item indisponível depois de eu já ter pago.",
    "Hambúrguer veio com a carne queimada. Não pedi de novo nesse lugar.",
    'Promoção "leve 3 pague 2" funcionou direitinho. Aproveitei pra encher a geladeira.',
    "Atrasou 1h e meia. Frio, encharcado, motoboy mal humorado. Pior pedido do mês.",
    "Pedi pra trocar um item depois que cheguei errado, suporte fez sem drama.",
    "Açaí do Açaí Mania é generoso e bem servido. Vale o que paga.",
    "Não consigo trocar endereço sem reinstalar o app. Bug antigo, ninguém arruma.",
    "Comida da Tia Nilda salva o dia. Sopa quentinha numa noite fria de pelotense.",
    "Cancelei o pedido depois de 1h de atraso. Reembolso veio em 7 dias úteis, sem briga.",
    "Bolo de cenoura desabou na embalagem. Estava gostoso, mas a cara tava triste.",
    "App caiu durante o pico do almoço. Pedi por outro app, perderam o cliente.",
    "Pedido completo, na hora certa, comida boa. Como tem que ser.",
    "Cobrança extra de R$ 8 que não constava na finalização. Suporte ignorou minha mensagem.",
    "Suporte automático não entendeu o problema. Tive que insistir muito pra cair pra um humano.",
    "Pagamento via Pix instantâneo no app, gostei da praticidade.",
    "Comida do Boteco do Zé impecável. Pediria de novo amanhã sem pensar.",
    "Entregador não usou máscara nem capacete direito. Reportei pelo app.",
    "Filé à parmegiana do Restaurante da Esquina é nota 10. Generoso, bem temperado.",
    "Cupom sumiu do meu carrinho na hora de pagar. Tive que pagar cheio. Frustrante demais.",
    "Atendente do RápidoJá ligou pra confirmar o cancelamento antes. Atencioso.",
    "Pizza fria, app travou pra avaliar, suporte demorou pra responder. Dia ruim no RápidoJá.",
    "Comida boa, mas a taxa de entrega comeu o desconto da promoção. Saiu igual.",
    "App melhorou bastante depois da última atualização. Mais rápido, menos bug.",
    "Sumiu meu pedido no fluxo do app. Suporte achou depois de 30 minutos, pelo menos resolveram.",
    "Suco do Verdurama é fresquinho. Pedido do almoço veio rápido também.",
    "Cobraram da minha conta, restaurante diz que não recebeu. Briga deles, problema meu.",
    "Login com biometria voltou a funcionar depois da atualização. Era o que faltava.",
    "Comida do Sushi Express decaiu muito de qualidade. Não é mais o que era.",
    "RápidoJá tá deixando a desejar. Atrasou, app travou, suporte sumiu. Tudo num pedido só.",
    "Promoção de frete grátis no horário do almoço salvou a semana. Boa estratégia do app.",
    "Mais um pedido atrasado. Restaurante diz que entregou pro motoboy, motoboy diz que pegou agora. Conversinha.",
    "Comida estava boa, atendimento simpático depois que reclamei do atraso. Equilibrou.",
    "Tô cansado de pagar caro pra esperar muito. Achei mais barato e mais rápido pegar no balcão.",
    "Pedido entregue na vizinha. Ela veio entregar pra mim, dei gorjeta pra ela. Pelo menos.",
    "Comida do dia foi excelente. Bem servido, bem temperado, fresquinho.",
    "Atrasou 30 min, mas o entregador foi gentil e a comida tava quente. Tolerável.",
    "Marmita gelada. Entregador disse que ficou esperando o restaurante 20 min. Não é justificativa.",
    "Atendente do chat resolveu em 3 minutos. Eficiente.",
    "Sumiu meu cupom no app na hora de pagar. Pagamento foi sem desconto. Não gostei.",
    "Refri veio quente. Motoboy disse que a moto não tem como gelar. Entendo, mas chato.",
    "App pediu avaliação 3 vezes seguidas do mesmo pedido. Irritante.",
    "Comida quente, entrega na hora marcada, atendimento via chat super atencioso. Top.",
    "Suporte humano e rápido. Salvaram o dia depois do erro do restaurante.",
    "Tá caro demais comparado com pegar no balcão. Pra ocasiões especiais, vai. Pra rotina, não.",
    "Comida do Tia Nilda é a melhor da plataforma. Sempre certo, sempre temperada do jeito que gosto.",
    "Atendente desligou o chat antes de resolver. Tive que reabrir e explicar tudo de novo.",
    "Não vou mais pedir aqui. Última vez foi atraso, comida fria, suporte demorando. Migro pra concorrência.",
    "Comi numa quarta à noite, comida do Boteco impecável. Recomendo o filé com fritas.",
    "Recurso de agendar pedido funciona bem. Uso pro café da manhã do escritório toda terça.",
    "Pedido sumiu, depois de 2h apareceu cancelado. Nem reembolso automático teve, tive que pedir.",
    "Comida boa, mas o app não me deixa avaliar. Buga toda vez que tento dar 5 estrelas.",
    "Pizza do Boteco gigante e bem montada. Vale cada centavo do que cobram.",
    "Marmita do Verdurama no horário, comida bem temperada. Sem reclamações hoje.",
]